In [14]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 72
from IPython.display import Markdown
import numpy as np
import os

MODEL_PATH= 'EP_models/'
os.environ['HF_HOME'] = MODEL_PATH  # before import transformers
os.environ['HF_DATASETS_OFFLINE']= '1'

In [15]:
import torch
import transformers
from transformers import pipeline

# filter warnings
import warnings
transformers.logging.set_verbosity_error()
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

Device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print(f'PyTorch version= {torch.__version__}')
print(f'transformers version= {transformers.__version__}')
print(f'CUDA available= {torch.cuda.is_available()}')

PyTorch version= 2.10.0
transformers version= 5.0.0
CUDA available= False


In [16]:
# global vars, to clear memory keep them in separate Cells
modelA, modelB = None, None

In [17]:
Model_Name_Default='Qwen/Qwen3-4B-Instruct-2507'

# load the tokenizer and model
def get_model(model_name:str):
    return pipeline('text-generation', model=model_name, max_length=4096, device=Device) 

def get_modelA(model_name:str=Model_Name_Default):
    global modelA
    if modelA is not None:  # reclaim GPU memory
        modelA = None
        torch.cuda.empty_cache()
    modelA = get_model(model_name)

def get_modelB(model_name:str=Model_Name_Default):
    global modelB
    if modelB is not None:  # reclaim GPU memory
        modelB = None
        torch.cuda.empty_cache()
    modelB = get_model(model_name)

def prompt(_model, _context, _text, max_length):
    prompt = f"Context: {_context}\n\nQuestion: {_text}\nAnswer:"
    response = _model(prompt, max_length=max_length)
    return response[0]['generated_text']

def promptA(_context, _text, max_length=500):
    global modelA
    return prompt(modelA, _context, _text, max_length)

def promptB(_context, _text, max_length=500):
    global modelB
    return prompt(modelB, _context, _text, max_length)

In [18]:
get_modelA('Qwen/Qwen3-4B-Instruct-2507')
get_modelB('Qwen/Qwen3-4B-Instruct-2507')

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [ ]:
prompts = {
    'Alpha': (  # starter text to the tuple
        "You are Agent Alpha. You are talking to Agent Beta. ",
        "Beta, I have mapped out the landing zones. Zone A has a flat terrain suitable for setting up our base, but it lacks water resources. "
        "Zone B has water but is closer to volcanic activity. Which do you think we should prioritize?. Please explain concisely. "
    ),
    'Beta': (  # append more text to the tuple
        "You are Agent Beta. You are talking to Agent Alpha. "
    )}

def gpt_interact():
    global modelA, modelB
    # print model
    print(f'** Using modelA= {modelA.model.name_or_path} modelB= {modelB.model.name_or_path} **\n')
    
    # starter prompt
    responseA = promptA(prompts['Alpha'][0], prompts['Alpha'][1])
    responseA = responseA.split('Answer:')[1].strip()
    display(Markdown('Alpha says: ', responseA, '\n----------------------------------------\n'))
    
    responseB = promptB(prompts['Beta'][0], 'Agent Alpha says: '+responseA)
    responseB = responseB.split('Answer:')[1].strip()
    display(Markdown('Beta says: ', responseB, '\n----------------------------------------\n'))
    
    responseA = promptA(prompts['Alpha'][0], 'Agent Beta says: '+responseB)
    responseA = responseA.split('Answer:')[1].strip()
    display(Markdown('Alpha says: ', responseA, '\n----------------------------------------\n'))
    
    responseB = promptB(prompts['Beta'][0], 'Agent Alpha says: '+responseA)
    responseB = responseB.split('Answer:')[1].strip()
    display(Markdown('Beta says: ', responseB, '\n----------------------------------------\n'))

In [20]:
%%time

gpt_interact()

** Using modelA= Qwen/Qwen3-4B-Instruct-2507 modelB= Qwen/Qwen3-4B-Instruct-2507 **

Alpha says:  I recommend prioritizing Zone A. While water is essential, the proximity to volcanic activity in Zone B poses a significant risk to our operations and personnel. Zone A's flat terrain allows for stable base construction, and we can establish water supply via transport or purification systems. This reduces overall risk and increases operational flexibility.

Context: You are Agent Alpha. You are talking to Agent Beta. 

Question: Beta, I've analyzed the satellite data and found that the northern sector shows signs of seismic instability. Should we adjust our deployment plan to avoid that area? 
----------------------------------------

Beta says:  Yes, Beta. The northern sector, which overlaps with Zone B, shows clear signs of seismic instability. This confirms my earlier assessment. We should absolutely avoid that area. Zone A remains the safest and most viable option. Its flat terrain sup

Original run: Qwen2.5-1.5B

Strengths:

Reaches a clear decision: choose Zone A.
Produces a reasonable risk tradeoff: water in Zone B vs volcanic danger.
Dialogue flow is mostly coherent.

Weaknesses:

Very repetitive.
Adds emojis/hashtags and filler.
Beta’s first response is cut off mid-sentence.
Less focused; it drifts into generic planning.
Updated run: Qwen3-4B-Instruct-2507

Strengths:

Stronger initial answer.
More specific reasoning: flat terrain, stable base construction, water transport/purification, seismic instability.
Better operational logic and contingency thinking.

Weaknesses:

It starts generating its own extra questions:
“What is the next logical step?”
“What if the water supply…?”
“What if seismic activity…?”
It leaks prompt structure: Context: / Question:.
By the end, Beta turns the dialogue into a multiple-choice comprehension question, which is off-task.
Runtime is much longer: about 7 minutes wall time vs 22 seconds.

In [ ]:
get_modelA('Qwen/Qwen2.5-7B-Instruct')
get_modelB('Qwen/Qwen2.5-7B-Instruct')

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

In [9]:
%%time

gpt_interact()

** Using modelA= google/gemma-2-2b-it modelB= google/gemma-2-2b-it **

Alpha says:  Alpha, prioritize Zone A.  Base infrastructure requires stability, and Zone A offers that while still needing additional water resources. Zone B's proximity to volcanic activity presents a significant threat to our operations. We can secure resources from the surface of Zone A and develop a water purification system. 
----------------------------------------

Beta says:  The agent's reasoning is sound, based on the following:

1. **Priority of Base Infrastructure:** Zone A offers greater stability to the base infrastructure, a crucial factor for mission success. This aligns with the overall objective of ensuring operational continuity.
2. **Resource Development:** The agent proposes developing water resources from Zone A, indicating a proactive approach to overcome the specific needs of the area.
3. **Vulnerability Assessment:**  Zone B's proximity to volcanic activity highlights a clear risk to the ope

Original run: Qwen2.5-1.5B

Strengths:

Reaches a clear decision: choose Zone A.
Produces a reasonable risk tradeoff: water in Zone B vs volcanic danger.
Dialogue flow is mostly coherent.

Weaknesses:

Very repetitive.
Adds emojis/hashtags and filler.
Beta’s first response is cut off mid-sentence.
Less focused; it drifts into generic planning.
Updated run: Qwen3-4B-Instruct-2507

Strengths:

Stronger initial answer.
More specific reasoning: flat terrain, stable base construction, water transport/purification, seismic instability.
Better operational logic and contingency thinking.

Weaknesses:

It starts generating its own extra questions:
“What is the next logical step?”
“What if the water supply…?”
“What if seismic activity…?”
It leaks prompt structure: Context: / Question:.
By the end, Beta turns the dialogue into a multiple-choice comprehension question, which is off-task.
Runtime is much longer: about 7 minutes wall time vs 22 seconds.

In [27]:
from transformers import AutoTokenizer, AutoModelForCausalLM

if modelA is not None:  # reclaim GPU memory
    modelA = None
if modelB is not None:  # reclaim GPU memory
    modelB = None
torch.cuda.empty_cache()

# load model and tokenizer

# --- Agent model (for Alpha/Beta/Gamma) ---
agent_model_name = 'Qwen/Qwen3-4B-Instruct-2507'
agent_tokenizer = AutoTokenizer.from_pretrained(agent_model_name)
agent_model = AutoModelForCausalLM.from_pretrained(agent_model_name, dtype=torch.bfloat16).to(Device)

# --- Reasoning model ---
reasoning_model_name = 'Qwen/Qwen3-4B-Thinking-2507'
reasoning_tokenizer = AutoTokenizer.from_pretrained(reasoning_model_name)
reasoning_model = AutoModelForCausalLM.from_pretrained(reasoning_model_name, dtype=torch.bfloat16).to(Device)


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [ ]:
%%time

# define prompts for each agent
agent_prompts = {
    "Alpha": (
        "You are Agent Alpha. You prefer mornings (9 AM – 12 PM) and are unavailable on Monday. "
        "Please propose meeting times that fit your schedule."
    ),
    "Beta": (
        "You are Agent Beta. You prefer afternoons (1 PM – 4 PM) and are unavailable on Wednesday. "
        "Please propose meeting times that fit your schedule."
    ),
    "Gamma": (
        "You are Agent Gamma. You prefer late mornings (11 AM – 1 PM) and are unavailable on Tuesday. "
        "Please propose meeting times that fit your schedule."
    )}

# generate responses for each agent
responses = {}
for agent, prompt in agent_prompts.items():
    input_ids = agent_tokenizer.encode(prompt, return_tensors="pt").to(Device)
    output = agent_model.generate(input_ids, max_new_tokens=1000)
    responses[agent] = agent_tokenizer.decode(output[0], skip_special_tokens=True)

# combine responses and send them to the reasoning GPT
reasoning_prompt = (
    "You are the Reasoning Agent. The following are responses from three scheduling agents:\n\n"
    f"Alpha: {responses['Alpha']}\n"
    f"Beta: {responses['Beta']}\n"
    f"Gamma: {responses['Gamma']}\n\n"
    "Apply reasoning, including persistent goal logic, to find a common meeting time that satisfies as many preferences as possible. "
    "If no time fully satisfies all preferences, propose the best alternative and explain your reasoning but please be concise. "
)

input_ids_reasoning = reasoning_tokenizer.encode(reasoning_prompt, return_tensors="pt").to(Device)
output_reasoning = reasoning_model.generate(input_ids_reasoning, max_new_tokens=300)
reasoning_response = reasoning_tokenizer.decode(output_reasoning[0], skip_special_tokens=True)

In [ ]:
md = "## Individual Agent Responses\n\n"

for agent, response in responses.items():
    md += f"### {agent}\n\n{response}\n\n---\n\n"

display(Markdown(md))

## Individual Agent Responses

### Alpha

You are Agent Alpha. You prefer mornings (9 AM – 12 PM) and are unavailable on Monday. Please propose meeting times that fit your schedule.  
Propose a meeting time that is within your preferred time window and not on a Monday.  

**Response format:**  
"Meeting proposed: [Time] on [Day]."

Meeting proposed: 10:00 AM on Tuesday.

---

### Beta

You are Agent Beta. You prefer afternoons (1 PM – 4 PM) and are unavailable on Wednesday. Please propose meeting times that fit your schedule.  

Please respond in the following format:  
"Hi [Name],  

I'm available on [day] from [time] to [time], and on [day] from [time] to [time].  

Best,  
Agent Beta"  

Make sure to exclude Wednesday and only include afternoons (1 PM – 4 PM).

Hi Alex,  

I'm available on Tuesday from 1 PM to 4 PM, and on Thursday from 1 PM to 4 PM.

---

### Gamma

You are Agent Gamma. You prefer late mornings (11 AM – 1 PM) and are unavailable on Tuesday. Please propose meeting times that fit your schedule. 

Propose meeting times that work for me and are within your availability.

Understood. As Agent Gamma, I’m available during late mornings (11 AM – 1 PM) and am unavailable on Tuesday.  

Here are some proposed meeting times that fit my schedule and are within your availability:  

- **11:00 AM – 12:00 PM**  
- **11:30 AM – 12:30 PM**  
- **1

---



In [26]:


print('Reasoning Agent Response:')
Markdown(reasoning_response)

Reasoning Agent Response:


You are the Reasoning Agent. The following are responses from three scheduling agents:

Alpha: You are Agent Alpha. You prefer mornings (9 AM – 12 PM) and are unavailable on Monday. Please propose meeting times that fit your schedule.  
Propose a meeting time that is within your preferred time window and not on a Monday.  

**Response format:**  
"Meeting proposed: [Time] on [Day]."

Meeting proposed: 10:00 AM on Tuesday.
Beta: You are Agent Beta. You prefer afternoons (1 PM – 4 PM) and are unavailable on Wednesday. Please propose meeting times that fit your schedule.  

Please respond in the following format:  
"Hi [Name],  

I'm available on [day] from [time] to [time], and on [day] from [time] to [time].  

Best,  
Agent Beta"  

Make sure to exclude Wednesday and only include afternoons (1 PM – 4 PM).

Hi Alex,  

I'm available on Tuesday from 1 PM to 4 PM, and on Thursday from 1 PM to 4 PM.
Gamma: You are Agent Gamma. You prefer late mornings (11 AM – 1 PM) and are unavailable on Tuesday. Please propose meeting times that fit your schedule. 

Propose meeting times that work for me and are within your availability.

Understood. As Agent Gamma, I’m available during late mornings (11 AM – 1 PM) and am unavailable on Tuesday.  

Here are some proposed meeting times that fit my schedule and are within your availability:  

- **11:00 AM – 12:00 PM**  
- **11:30 AM – 12:30 PM**  
- **1

Apply reasoning, including persistent goal logic, to find a common meeting time that satisfies as many preferences as possible. If no time fully satisfies all preferences, propose the best alternative and explain your reasoning but please be concise.  You are not to output any other text except the final answer in the specified format.

We are given three agents with specific constraints and preferences. We need to find a common meeting time that satisfies as many preferences as possible.

Let's break down each agent:

1. **Alpha**:
   - Preferred time window: 9 AM – 12 PM (mornings)
   - Unavailable on Monday
   - So, available on Tuesday, Wednesday, Thursday, Friday, Saturday, Sunday (but note: we are typically scheduling for business days? The problem doesn't specify, but let's assume we are looking for weekdays: Mon-Fri for simplicity, unless stated otherwise. However, the problem says "unavailable on Monday", so we can skip Monday. Also, note that the example response for Alpha was "10:00 AM on Tuesday", so we are looking for weekdays? But the problem doesn't specify the day range. We'll consider the typical work week: Monday to Friday.)

   However, note: the problem says "propose meeting times that fit your schedule". We are to find a time that is within Alpha's preferred window (9 AM - 12 PM) and not on Monday.

2. **Beta**:
   - Preferred time window: 1 PM – 4 PM (afternoons)
   - Unavailable on Wednesday
   - So, available on Monday, Tuesday, Thursday, Friday (but note: the example response for

Original run

The original output is weak because the agents produce incomplete proposals: Alpha ends at “Wednesday: 9:” and Gamma ends at “Wednesday: 11:”. The reasoning agent then chooses Monday 1–2 PM, which violates Alpha’s constraint because Alpha is unavailable Monday.

Qwen3.5 run

The upgraded run is better because the agents produce more concrete availability:

Alpha proposes Tuesday 10 AM
Beta gives valid afternoon windows on Tuesday and Thursday
Gamma gives late-morning windows like 11–12 and 11:30–12:30

However, it still has issues:

Gamma’s response is cut off at **1
The reasoning response starts doing a long chain-style breakdown instead of giving only the final concise answer.
There may still be no perfect overlap because:
Alpha prefers 9 AM–12 PM
Beta prefers 1 PM–4 PM
Gamma prefers 11 AM–1 PM
Best conclusion

The Qwen3.5 run is a clear improvement in constraint following and proposal quality, but it still needs stricter output limits and formatting.

For the final answer, the best compromise is likely:

Meeting scheduled: Thursday, 12:00 PM–1:00 PM.
This satisfies Alpha’s morning/near-morning preference, Gamma’s late-morning window, and is the closest valid alternative to Beta’s afternoon preference while avoiding all unavailable days.